In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

def prepare():
    module_path = os.path.abspath(os.path.join('../','../'))
    if module_path not in sys.path:
        sys.path.append(module_path)

In [ ]:
import torch
import numpy as np
prepare()
from exp_labelcert_binaryclass import run

In [ ]:
model_params = dict(
    label = "GCN", 
    model = "GCN", 
    normalization = "row_normalization",
    activation = "relu",
    depth = 1,
    regularizer = 0.075,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.01,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license = "path/to/your/gurobi/license"
)

In [ ]:
data_params = dict(
    dataset = "cora_ml_binary", # use "citeseer_binary" for citeseer
    learning_setting = "transductive", 
    specification = dict(
        n_per_class = 5,
        fraction_test = 0.01,
        data_dir = "./data",
        make_undirected = True,
        binary_attr = False,
        balance_test = True,
    )
)

In [ ]:
# import pandas as pd
# import time

# seeds = [0, 1, 2, 3, 4]
# delta = 0.0
# certificate_params["delta"] = delta

# metrics = [
#     "accuracy_test",
#     "accuracy_trn",
#     "accuracy_cert_pois_robust",
#     "accuracy_cert_pois_unrobust",
#     "delta",
# ]

# summary = []

# for seed in seeds:
#     start_time = time.time()
#     result = run(data_params, model_params, certificate_params, verbosity_params, other_params, seed)
#     end_time = time.time()
#     runtime = round(end_time - start_time, 2)
    
#     summary.append({k: result[k] for k in metrics} | {"runtime": runtime})
    
    
# df = pd.DataFrame(summary, index=[f"Seed {s}" for s in seeds])
# df.index.name = "seed"
# df.to_csv(f'results/samplewise/coramlb-{delta:.2f}.csv', index=True)
# df

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path

module_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added '{module_path}' to sys.path to find project modules.")

from exp_labelcert_collective import run

model_params = {
    'label': 'GCN',
    'model': 'GCN',
    'normalization': 'row_normalization',
    'activation': 'relu',
    'depth': 1,
    'regularizer': 0.075, 
    'pred_method': 'svm',
    'bias': False,
    'alpha_tol': 1e-4,
    'solver': 'qplayer'
}

data_params = {
    'dataset': 'cora_ml_binary',
    'learning_setting': 'transductive',
    'specification': {
        'n_per_class': 5, 
        'fraction_test': 0.01,
        'data_dir': './data',
        'make_undirected': True,
        'binary_attr': False,
        'balance_test': True
    }
}

other_params = {
    'device': '0',
    'dtype': torch.float64,
    'allow_tf32': False,
    'path_gurobi_license':'/mnt/c/Users/emiel/gurobi.lic'
}

verbosity_params = {'debug_lvl': 'warning'}


eps_coarse = np.linspace(0.00, 0.30, 16).tolist()
eps_fine = np.linspace(0.13, 0.18, 26).tolist()
epsilons = sorted(set(eps_coarse + eps_fine))
seeds = range(10)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
output_filename = "final_gcn_on_coramlb_collective_data.csv"
output_path = log_dir / output_filename


print("--- Starting Data Generation for GCN on Cora-MLb (Collective) ---")
print(f"Running {len(epsilons)} epsilon points across {len(seeds)} seeds...")
all_results = []
start_time_total = time.time()

for seed_val in seeds:
    print(f"\nProcessing Seed: {seed_val}/{len(seeds)-1}...")
    data_params['specification']['seed'] = int(seed_val)
    
    for i, eps in enumerate(epsilons):
        certificate_params = {'delta': float(eps), 'n_test': 50}
        
        out = run(
            data_params=data_params,
            model_params=model_params,
            certificate_params=certificate_params,
            verbosity_params=verbosity_params,
            other_params=other_params,
            seed=int(seed_val)
        )
        
        out.update({'delta': float(eps), 'seed': int(seed_val)})
        all_results.append(out)
        
        print(f"  ({i+1}/{len(epsilons)}) ε={eps:.4f} | Robust Ratio: {out.get('accuracy_cert_pois_robust', 'N/A'):.4f} | Gurobi Nodes: {out.get('gurobi_node_count', 'N/A')}")

df_final = pd.DataFrame(all_results)
df_final.to_csv(output_path, index=False)

end_time_total = time.time()
print(f"\n--- ✅ Experiment Complete ---")
print(f"Total runtime: {(end_time_total - start_time_total) / 60:.2f} minutes")
print(f"Final data for {len(df_final)} runs saved to '{output_path}'")


In [ ]:
#2
import os, ast, numpy as np, pandas as pd, matplotlib.pyplot as plt, networkx as nx
from pathlib import Path
from src.data import get_cora_ml_binary
df_final = pd.DataFrame(all_results)
df_final.to_csv(output_path, index=False)

end_time_total = time.time()
print(f"\n--- ✅ Experiment Complete ---")
print(f"Total runtime: {(end_time_total - start_time_total) / 60:.2f} minutes")
print(f"Final data for {len(df_final)} runs saved to '{output_path}'")

print("\n--- Starting Analysis and Plotting ---")

df = df_final.copy()

data_spec = data_params["specification"]
cora_data = get_cora_ml_binary(
    n_per_class=data_spec["n_per_class"],
    fraction_test=data_spec["fraction_test"],
    data_dir=data_spec["data_dir"],
    make_undirected=data_spec["make_undirected"],
    seed=data_spec["seed"]
)
A = cora_data.adjacency_matrix
G = nx.from_numpy_array(A.toarray() if hasattr(A, 'toarray') else A)
centrality = nx.betweenness_centrality(G)

def parse_flip(x):
    if isinstance(x, str):
        parsed = ast.literal_eval(x)
    else:
        parsed = x
    return np.array(parsed, dtype=int)

df["y_flip_arr"] = df["y_flip"].apply(parse_flip)

#  avg betweenness for each row
df["avg_bc_flipped"] = df["y_flip_arr"].apply(
    lambda arr: np.mean([centrality[i] for i in np.where(arr==1)[0]]) if arr.sum()>0 else 0.0
)

# group by ε and compute mean±std
group = df.groupby("delta")
stats = group.agg({
    "accuracy_cert_pois_robust": ["mean", "std"],
    "gurobi_node_count":         ["mean", "std"],
    "avg_bc_flipped":            ["mean", "std"],
}).reset_index()
stats.columns = [
    "delta",
    "robust_mean","robust_std",
    "nodes_mean","nodes_std",
    "bc_mean","bc_std"
]

# Plot with error bands
fig, axs = plt.subplots(1, 3, figsize=(18,4))

#  Robustness Plateau
axs[0].plot(stats.delta, stats.robust_mean, "-o")
axs[0].fill_between(stats.delta,
                    stats.robust_mean - stats.robust_std,
                    stats.robust_mean + stats.robust_std,
                    alpha=0.2)
axs[0].set(title="A) Robustness Plateau", xlabel="ε", ylabel="Certified Robustness")
axs[0].grid(True)

#  Complexity Cliff (log scale)
axs[1].plot(stats.delta, stats.nodes_mean, "-o")
axs[1].fill_between(stats.delta,
                    stats.nodes_mean - stats.nodes_std,
                    stats.nodes_mean + stats.nodes_std,
                    alpha=0.2)
axs[1].set(title="B) Complexity Cliff", xlabel="ε", ylabel="NodeCount")
axs[1].set_yscale("log")
axs[1].grid(True, which="both", linestyle="--")

#  Strategy Shift
axs[2].plot(stats.delta, stats.bc_mean, "-o")
axs[2].fill_between(stats.delta,
                    stats.bc_mean - stats.bc_std,
                    stats.bc_mean + stats.bc_std,
                    alpha=0.2)
axs[2].set(title="C) Adversarial Strategy Shift", 
           xlabel="ε", ylabel="Avg Betweenness (flipped)")
axs[2].grid(True)

plt.tight_layout()

plot_filename = "cora_gcn_complexity_cascade_analysis.png"
plot_path = log_dir / plot_filename
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Plot saved to: {plot_path}")

plt.show()

print("--- ✅ Analysis Complete ---")
print(f"Summary statistics:")
print(f"  - Total epsilon points: {len(stats)}")
print(f"  - Epsilon range: {stats.delta.min():.4f} to {stats.delta.max():.4f}")
print(f"  - Max node count: {stats.nodes_mean.max():.0f}")
print(f"  - Robustness range: {stats.robust_mean.min():.4f} to {stats.robust_mean.max():.4f}")
